### 1. 라이브러리 불러오기

In [2]:
import os
import pandas as pd
import numpy as np
import datetime
import holidays
from flaml import AutoML
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

import warnings
warnings.filterwarnings("ignore")

### 2. 경로 설정

In [3]:
# 경로 설정
HOME = os.getcwd()
trainpath = os.path.join(HOME, 'data', 'train.csv')
testpath = os.path.join(HOME, 'data', 'test.csv')
subpath = os.path.join(HOME, 'data', 'sample_submission.csv')
interpath = os.path.join(HOME, 'data', 'international_trade.csv')

# 파일 읽기
train_df = pd.read_csv(trainpath)
test_df = pd.read_csv(testpath)
sub_df = pd.read_csv(subpath)
inter_df = pd.read_csv(interpath)

### 3. 데이터 전처리

In [4]:
df = pd.concat([train_df, test_df]).reset_index(drop = True)
df.rename(columns = {'supply(kg)' : 'supply', 'price(원/kg)' : 'price'}, inplace = True)

# 날짜 관련 특성
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['year_month'] = df['timestamp'].dt.to_period('M').astype(str)
df['year'] = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month
df['day'] = df['timestamp'].dt.day
df['dow'] = df['timestamp'].dt.dayofweek
df['week'] = df['timestamp'].dt.isocalendar().week.astype(int)
mask = (df['timestamp'].dt.month == 12) & (df['week'] == 1)
df.loc[mask, 'week'] = 52

kr_holidays = holidays.KR()
df['holiday'] = df['timestamp'].apply(lambda x: 1 if x in kr_holidays else 0)

# 무역 데이터 처리 (연도별 환율)
target_items = ['감귤', '당근', '양배추', '순무', '꽃양배추와 브로콜리(broccoli)']
inter_df = inter_df[inter_df['품목명'].isin(target_items)]
inter_df.loc[inter_df['품목명']=='꽃양배추와 브로콜리(broccoli)', '품목명'] = '브로콜리'
inter_df.loc[inter_df['품목명']=='순무', '품목명'] = '무'

exchange_rates = {2019: 1165, 2020: 1180, 2021: 1144, 2022: 1292, 2023: 1305}
inter_df['trade_timestamp'] = pd.to_datetime(inter_df['기간'])
inter_df['rate'] = inter_df['trade_timestamp'].dt.year.map(exchange_rates)
inter_df['수입 금액'] = inter_df['수입 금액'] * 1000 * inter_df['rate']

name_dict = {'감귤':'TG' ,'브로콜리':'BC' ,'무':'RD' ,'당근':'CR' ,'양배추':'CB'}
inter_df['품목명'] = inter_df['품목명'].map(name_dict)
inter_df['순수입량'] = inter_df['수입 중량'] - inter_df['수출 중량']
inter_df['kg당 금액'] = inter_df['수입 금액'] / (inter_df['수입 중량'] + 1)
inter_df['year_month_lagged'] = (inter_df['trade_timestamp'] + pd.DateOffset(months=1)).dt.to_period('M').astype(str)

inter_to_merge = inter_df[['year_month_lagged', '품목명', '수출 중량', '수입 중량', '순수입량', 'kg당 금액']].copy()
inter_to_merge.rename(columns={'year_month_lagged': 'year_month', '품목명': 'item'}, inplace=True)
df = pd.merge(df, inter_to_merge, on=['year_month', 'item'], how='left')
df[['수출 중량', '수입 중량', '순수입량', 'kg당 금액']] = df[['수출 중량', '수입 중량', '순수입량', 'kg당 금액']].fillna(0)

categorical_cols = ['item', 'corporation', 'location']
df_encoded = pd.get_dummies(df, columns=categorical_cols)

# 가격 0 제외 학습 데이터 생성
train_df = df_encoded[(~df_encoded['price'].isnull()) & (df_encoded['price'] > 0)]
test_df = df_encoded[df_encoded['price'].isnull()]

drop_cols = ['ID', 'timestamp', 'supply', 'price', 'year_month']
x_cols = [col for col in train_df.columns if col not in drop_cols]

### 4. 모델 설정

In [5]:
def run_automl(train_data, x_cols, time_budget=60):
    automl = AutoML()
    settings = {
        "time_budget": time_budget,  # 초 단위
        "metric": 'mae',
        "task": 'regression',
        "seed": 42,
        "verbose": 0
    }
    automl.fit(X_train=train_data[x_cols], y_train=train_data['price'], **settings)
    return automl

# TG 그룹 AutoML
print("Training TG Group AutoML...")
train_tg = train_df[train_df['item_TG'] == 1]
automl_tg = run_automl(train_tg, x_cols, time_budget=120)

# Others 그룹 AutoML
print("Training Others Group AutoML...")
train_others = train_df[train_df['item_TG'] == 0]
automl_others = run_automl(train_others, x_cols, time_budget=120)

print("Best model for TG:", automl_tg.best_estimator)
print("Best model for Others:", automl_others.best_estimator)

Training TG Group AutoML...
Training Others Group AutoML...
Best model for TG: xgboost
Best model for Others: lgbm


### 5. 결과 저장


In [6]:
# 예측
test_tg = test_df[test_df['item_TG'] == 1].copy()
test_others = test_df[test_df['item_TG'] == 0].copy()

test_tg['pred'] = automl_tg.predict(test_tg[x_cols])
test_others['pred'] = automl_others.predict(test_others[x_cols])

# 음수 보정 및 일요일 처리
for t_df in [test_tg, test_others]:
    t_df.loc[t_df['pred'] < 0, 'pred'] = 0
    t_df.loc[t_df['dow'] == 6, 'pred'] = 0

# 합치기 및 저장
final_test = pd.concat([test_tg, test_others])
final_submission = pd.merge(sub_df[['ID']], final_test[['ID', 'pred']], on='ID', how='left')
final_submission.rename(columns={'pred': 'price(원/kg)'}, inplace=True)
final_submission.to_csv('submission8.csv', index=False)
print("submission8.csv saved using AutoML (FLAML)")

submission8.csv saved using AutoML (FLAML)
